In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

###Creazione Tabella gold

In [0]:
silver_df = spark.read.table("silver_breweries")

gold_table = "gold_breweries"

if not spark.catalog.tableExists(gold_table):
  (silver_df
          .withColumn("brewery_sk", F.expr("uuid()")) #F.expr("uuid()") - identificatore univoco casuale in formato 36 caratteri
          .withColumn("valid_from", F.current_timestamp())
          .withColumn("valid_to", F.lit(None).cast("timestamp"))
          .withColumn("current", F.lit(True))
          .write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(gold_table)
  )


###Merge SCD2 Manuale

In [0]:
delta_table_gold = DeltaTable.forName(spark, gold_table)

delta_table_gold.alias("target").merge(
  silver_df.alias("source"),
  "target.id = source.id AND target.current = true",
) \
.whenMatchedUpdate(
  condition =
    """
    NOT
      (
      target.name <> source.name OR
      target.brewery_type <> source.brewery_type OR
      target.address_1 <> source.address_1 OR
      target.address_2 <> source.address_2 OR
      target.address_3 <> source.address_3 OR
      target.city <> source.city OR
      target.country <> source.country OR
      target.latitude <> source.latitude OR
      target.longitude <> source.longitude OR
      target.phone <> source.phone OR
      target.postal_code <> source.postal_code OR
      target.state <> source.state OR
      target.state_province <> source.state_province OR
      target.street <> source.street OR
      target.website_url <> source.website_url  
      )
    """,
  set = {
    "valid_to": F.current_timestamp(),
    "current": "false"
    }
) \
.whenNotMatchedInsert(	
  values={
    "brewery_sk": F.expr("uuid()"),
    "id": "source.id",
    "name": "source.name",
    "brewery_type": "source.brewery_type",
    "address_1": "source.address_1",
    "address_2": "source.address_2",
    "address_3": "source.address_3",
    "city": "source.city",
    "country": "source.country",
    "latitude": "source.latitude",
    "longitude": "source.longitude",
    "phone": "source.phone",
    "postal_code": "source.postal_code",
    "state": "source.state",
    "state_province": "source.state_province",
    "street": "source.street",
    "website_url": "source.website_url",
    "valid_from": F.current_timestamp(),
    "valid_to": F.lit(None).cast("timestamp"),
    "current": F.lit(True)
    }
).execute()

In [0]:
gold_table_order = "gold_breweries"

(
spark
    .read
    .table(gold_table_order)
    .select(
        "brewery_sk",          # surrogate prima
        "id",
        "name",
        "brewery_type",
        "address_1",
        "address_2",
        "address_3",
        "city",
        "state",
        "country",
        "postal_code",
        "phone",
        "website_url",
        "valid_from",
        "valid_to",
        "current",
        "latitude",
        "longitude",
        "state_province",
        "street"
    )
    .orderBy(F.col("brewery_sk").asc())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

###Verfica SCD2

In [0]:
df_gold = spark.read.table(gold_table_order)
display(
    df_gold
        .filter(F.col("current") == True)
        .orderBy(F.col("brewery_sk").asc())
)

###Aggregazioni

####Numero Brewery per Stato

In [0]:
(
  spark.table(gold_table_order)
  .filter("current = true")
  .groupBy("state")
  .count()
  .withColumnRenamed("count", "num_breweries")
  .orderBy(F.desc("num_breweries"))
  .write
  .mode("overwrite")
  .format("delta")
  .saveAsTable("gold_table_breweries_by_state")
)

In [0]:
gold_selection = spark.sql("""SELECT * FROM gold_breweries""")
display(gold_selection)

In [0]:
gold_selection_aggregate = spark.sql("""SELECT * FROM gold_table_breweries_by_state""")
display(gold_selection_aggregate)